# TAC-LAnoBERT v2: Complete Evaluation & Comparison

**Purpose**: Compare all optimization phases and baselines

---

## 📊 Models Compared

### Baselines
1. **LAnoBERT** - Original BERT-based log anomaly detection
2. **TAC-LAnoBERT (original)** - Time2Vec + Memory Queue (Mahalanobis)
3. **TAC v2 (2-epoch)** - v2 improvements (early detection, temporal features, data aug)

### Optimizations
4. **Phase 1 (KNN+PCA)** - KNN distance + PCA reduction + alpha=0.85
5. **Phase 2 (Projection Head)** - Learned 768→256→64 projection for early detection

---

## 🎯 Comparison Metrics

### Standard Performance
- F1-Score, Precision, Recall
- AUROC (Area Under ROC Curve)
- FPR (False Positive Rate)
- False Positives (absolute count)

### Early Detection
- **DLT** (Detection Lead Time) - seconds before failure
- **EWR** (Early Warning Rate) - % of anomalies detected early

### Operational Impact
- Alert volume (total alerts per day/week)
- Alert fatigue reduction
- ROI estimation (cost savings)

---

## 1. Setup

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Imports complete")

In [ ]:
# Check environment
if not os.getcwd().endswith('TAC-LAnoBERT-y'):
    if os.path.exists('TAC-LAnoBERT-y'):
        %cd TAC-LAnoBERT-y
    else:
        print("⚠️  Not in project directory")
        print("   Run training/evaluation notebooks first")

print(f"Working directory: {os.getcwd()}")
print("✅ Environment ready")

## 2. Load All Results

In [ ]:
def parse_report(report_path):
    """Parse TAC report file for metrics"""
    if not os.path.exists(report_path):
        return None
    
    with open(report_path, 'r') as f:
        content = f.read()
    
    metrics = {}
    
    # Basic metrics
    patterns = {
        'auroc': r'AUROC:\s+([0-9.e+-]+)',
        'f1': r'best_F1:\s+([0-9.e+-]+)',
        'precision': r'best_precision:\s+([0-9.e+-]+)',
        'recall': r'best_recall:\s+([0-9.e+-]+)',
        'threshold': r'best_threshold:\s+([0-9.e+-]+)',
    }
    
    for key, pattern in patterns.items():
        if m := re.search(pattern, content):
            metrics[key] = float(m.group(1))
    
    # Confusion matrix
    cm_pattern = r'confusion_matrix:.*?\[\[\s*(\d+)\s+(\d+)\s*\]\s*\[\s*(\d+)\s+(\d+)\s*\]\]'
    if m := re.search(cm_pattern, content, re.DOTALL):
        tn, fp, fn, tp = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        metrics.update({
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        })
    
    # Early detection metrics
    if m := re.search(r'DLT_mean:\s+([0-9.e+-]+)', content):
        metrics['dlt_mean'] = float(m.group(1))
    if m := re.search(r'EWR:\s+([0-9.e+-]+)', content):
        metrics['ewr'] = float(m.group(1))
    
    return metrics if metrics else None

print("✅ Parser function defined")

In [ ]:
# Find all available results
results = {}

# Define models to search for
models = {
    'LAnoBERT (baseline)': 'outputs/BGL_lanobert/results/BGL_error_mean_report.txt',
    'TAC (original)': 'outputs/BGL_tac/results/BGL_tac_hybrid_report.txt',
    'TAC v2 (2-epoch)': 'outputs/BGL_tac_v2_2epochs/results/*_report.txt',
    'Phase 1 (KNN+PCA)': 'outputs/BGL_tac_v2_optimized/results/*_report.txt',
    'Phase 2 (Projection)': 'outputs/BGL_tac_v2_phase2/results/*_report.txt',
}

print("=" * 70)
print("SEARCHING FOR RESULTS")
print("=" * 70)

for model_name, pattern in models.items():
    if '*' in pattern:
        # Glob pattern
        import glob
        files = glob.glob(pattern)
        if files:
            metrics = parse_report(files[0])
            if metrics:
                results[model_name] = metrics
                print(f"✅ {model_name:<25} Found")
            else:
                print(f"⚠️  {model_name:<25} File found but parse failed")
        else:
            print(f"❌ {model_name:<25} Not found")
    else:
        # Direct path
        if os.path.exists(pattern):
            metrics = parse_report(pattern)
            if metrics:
                results[model_name] = metrics
                print(f"✅ {model_name:<25} Found")
            else:
                print(f"⚠️  {model_name:<25} File found but parse failed")
        else:
            print(f"❌ {model_name:<25} Not found")

print("\n" + "=" * 70)
print(f"LOADED {len(results)}/{len(models)} MODELS")
print("=" * 70)

if len(results) == 0:
    print("\n❌ No results found!")
    print("   Run training/evaluation notebooks first.")
else:
    print(f"\nAvailable models:")
    for name in results.keys():
        print(f"  • {name}")

## 3. Performance Comparison Table

In [ ]:
# Create comparison DataFrame
if results:
    # Define metrics to compare
    metric_cols = ['f1', 'precision', 'recall', 'auroc', 'fpr', 'fp', 'fn', 'dlt_mean', 'ewr']
    
    # Build DataFrame
    df_data = {}
    for model_name, metrics in results.items():
        df_data[model_name] = {col: metrics.get(col) for col in metric_cols}
    
    df = pd.DataFrame(df_data).T
    
    # Format for display
    df_display = df.copy()
    
    # Format percentages
    for col in ['f1', 'precision', 'recall', 'auroc', 'ewr']:
        if col in df_display.columns:
            df_display[col] = df_display[col].apply(lambda x: f"{x:.6f}" if pd.notna(x) else "N/A")
    
    # Format FPR
    if 'fpr' in df_display.columns:
        df_display['fpr'] = df_display['fpr'].apply(lambda x: f"{x*100:.4f}%" if pd.notna(x) else "N/A")
    
    # Format integers
    for col in ['fp', 'fn']:
        if col in df_display.columns:
            df_display[col] = df_display[col].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    
    # Format DLT
    if 'dlt_mean' in df_display.columns:
        df_display['dlt_mean'] = df_display['dlt_mean'].apply(
            lambda x: f"{x:.1f}s ({x/60:.1f}m)" if pd.notna(x) else "N/A"
        )
    
    # Rename columns for display
    df_display.columns = [
        'F1', 'Precision', 'Recall', 'AUROC', 'FPR', 
        'False Pos', 'False Neg', 'DLT (mean)', 'EWR'
    ]
    
    print("=" * 140)
    print("PERFORMANCE COMPARISON TABLE")
    print("=" * 140)
    print()
    print(df_display.to_string())
    print()
    print("=" * 140)
else:
    print("⚠️  No results to display")

## 4. Improvement Analysis

In [ ]:
# Calculate improvements relative to baseline
if results and len(results) >= 2:
    # Use LAnoBERT or first result as baseline
    baseline_name = 'LAnoBERT (baseline)' if 'LAnoBERT (baseline)' in results else list(results.keys())[0]
    baseline = results[baseline_name]
    
    print("=" * 70)
    print(f"IMPROVEMENTS OVER {baseline_name.upper()}")
    print("=" * 70)
    
    improvement_metrics = ['f1', 'precision', 'recall', 'auroc', 'fpr', 'fp', 'dlt_mean', 'ewr']
    
    for model_name, metrics in results.items():
        if model_name == baseline_name:
            continue
        
        print(f"\n{model_name}:")
        print("-" * 70)
        
        for metric in improvement_metrics:
            if metric in baseline and metric in metrics:
                base_val = baseline[metric]
                curr_val = metrics[metric]
                
                if base_val is not None and curr_val is not None:
                    if metric in ['fpr', 'fp', 'fn']:  # Lower is better
                        if base_val > 0:
                            improvement = (base_val - curr_val) / base_val * 100
                            status = "✅" if improvement > 0 else "❌"
                        else:
                            improvement = 0
                            status = "≈"
                    else:  # Higher is better
                        if base_val > 0:
                            improvement = (curr_val - base_val) / base_val * 100
                            status = "✅" if improvement > 0 else "❌"
                        else:
                            improvement = 0
                            status = "≈"
                    
                    # Format output
                    if metric == 'fp':
                        print(f"  {status} {metric.upper():<12} {base_val:>8,} → {curr_val:>8,}  ({improvement:+.2f}%)")
                    elif metric in ['fpr', 'ewr']:
                        print(f"  {status} {metric.upper():<12} {base_val*100:>8.4f}% → {curr_val*100:>8.4f}%  ({improvement:+.2f}%)")
                    elif metric == 'dlt_mean':
                        print(f"  {status} {metric.upper():<12} {base_val:>8.1f}s → {curr_val:>8.1f}s  ({improvement:+.2f}%)")
                    else:
                        print(f"  {status} {metric.upper():<12} {base_val:>8.6f} → {curr_val:>8.6f}  ({improvement:+.2f}%)")
    
    print("\n" + "=" * 70)
else:
    print("⚠️  Need at least 2 models for comparison")

## 5. Visualizations

In [ ]:
# Plot key metrics comparison
if results:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('TAC-LAnoBERT v2: Performance Comparison', fontsize=16, fontweight='bold')
    
    # Prepare data
    models = list(results.keys())
    
    # 1. F1-Score
    ax = axes[0, 0]
    f1_scores = [results[m].get('f1', 0) for m in models]
    bars = ax.bar(range(len(models)), f1_scores, color='steelblue')
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title('F1-Score Comparison', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=9)
    ax.set_ylim([min(f1_scores) * 0.95 if f1_scores else 0, 1.0])
    ax.grid(axis='y', alpha=0.3)
    # Add values on bars
    for i, v in enumerate(f1_scores):
        ax.text(i, v + 0.005, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    
    # 2. AUROC
    ax = axes[0, 1]
    auroc_scores = [results[m].get('auroc', 0) for m in models]
    bars = ax.bar(range(len(models)), auroc_scores, color='forestgreen')
    ax.set_ylabel('AUROC', fontsize=12)
    ax.set_title('AUROC Comparison', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=9)
    ax.set_ylim([min(auroc_scores) * 0.95 if auroc_scores else 0, 1.0])
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(auroc_scores):
        ax.text(i, v + 0.005, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    
    # 3. False Positives
    ax = axes[0, 2]
    fp_counts = [results[m].get('fp', 0) for m in models]
    bars = ax.bar(range(len(models)), fp_counts, color='coral')
    ax.set_ylabel('False Positives', fontsize=12)
    ax.set_title('False Positives (Lower is Better)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(fp_counts):
        ax.text(i, v + max(fp_counts)*0.02, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
    
    # 4. FPR
    ax = axes[1, 0]
    fpr_values = [results[m].get('fpr', 0) * 100 for m in models]
    bars = ax.bar(range(len(models)), fpr_values, color='tomato')
    ax.set_ylabel('FPR (%)', fontsize=12)
    ax.set_title('False Positive Rate (Lower is Better)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(fpr_values):
        ax.text(i, v + max(fpr_values)*0.02, f'{v:.4f}%', ha='center', va='bottom', fontsize=9)
    
    # 5. DLT (Early Detection)
    ax = axes[1, 1]
    dlt_values = [results[m].get('dlt_mean', 0) / 60 for m in models]  # Convert to minutes
    bars = ax.bar(range(len(models)), dlt_values, color='mediumpurple')
    ax.set_ylabel('DLT (minutes)', fontsize=12)
    ax.set_title('Detection Lead Time (Higher is Better)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(dlt_values):
        if v > 0:
            ax.text(i, v + max(dlt_values)*0.02, f'{v:.1f}m', ha='center', va='bottom', fontsize=9)
    
    # 6. EWR (Early Warning Rate)
    ax = axes[1, 2]
    ewr_values = [results[m].get('ewr', 0) * 100 for m in models]
    bars = ax.bar(range(len(models)), ewr_values, color='gold')
    ax.set_ylabel('EWR (%)', fontsize=12)
    ax.set_title('Early Warning Rate (Higher is Better)', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in models], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(ewr_values):
        if v > 0:
            ax.text(i, v + max(ewr_values)*0.02, f'{v:.2f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Save figure
    os.makedirs('outputs/comparison', exist_ok=True)
    fig.savefig('outputs/comparison/performance_comparison.png', dpi=300, bbox_inches='tight')
    print("✅ Figure saved to: outputs/comparison/performance_comparison.png")
else:
    print("⚠️  No results to visualize")

In [ ]:
# Improvement heatmap (relative to baseline)
if results and len(results) >= 2:
    baseline_name = 'LAnoBERT (baseline)' if 'LAnoBERT (baseline)' in results else list(results.keys())[0]
    baseline = results[baseline_name]
    
    # Calculate percentage improvements
    metrics_to_compare = ['f1', 'precision', 'recall', 'auroc', 'fpr', 'fp', 'dlt_mean', 'ewr']
    improvement_data = []
    
    for model_name in results.keys():
        if model_name == baseline_name:
            continue
        
        model_improvements = []
        for metric in metrics_to_compare:
            base_val = baseline.get(metric)
            curr_val = results[model_name].get(metric)
            
            if base_val is not None and curr_val is not None and base_val != 0:
                if metric in ['fpr', 'fp']:  # Lower is better
                    improvement = (base_val - curr_val) / base_val * 100
                else:  # Higher is better
                    improvement = (curr_val - base_val) / base_val * 100
            else:
                improvement = 0
            
            model_improvements.append(improvement)
        
        improvement_data.append(model_improvements)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    
    model_names = [m for m in results.keys() if m != baseline_name]
    metric_labels = ['F1', 'Precision', 'Recall', 'AUROC', 'FPR↓', 'FP↓', 'DLT', 'EWR']
    
    im = ax.imshow(improvement_data, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=50)
    
    # Set ticks
    ax.set_xticks(np.arange(len(metric_labels)))
    ax.set_yticks(np.arange(len(model_names)))
    ax.set_xticklabels(metric_labels, fontsize=11)
    ax.set_yticklabels(model_names, fontsize=11)
    
    # Add values
    for i in range(len(model_names)):
        for j in range(len(metric_labels)):
            text = ax.text(j, i, f'{improvement_data[i][j]:.1f}%',
                          ha="center", va="center", color="black", fontsize=9, fontweight='bold')
    
    ax.set_title(f'Performance Improvement over {baseline_name}\n(Positive = Better)', 
                fontsize=14, fontweight='bold', pad=20)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Improvement (%)', fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    # Save
    fig.savefig('outputs/comparison/improvement_heatmap.png', dpi=300, bbox_inches='tight')
    print("✅ Heatmap saved to: outputs/comparison/improvement_heatmap.png")
else:
    print("⚠️  Need at least 2 models for improvement analysis")

## 6. ROI Analysis

Estimate operational impact and cost savings.

In [ ]:
# ROI calculation
if results:
    print("=" * 70)
    print("ROI ANALYSIS (Estimated)")
    print("=" * 70)
    
    # Assumptions
    COST_PER_FP = 100  # USD - engineer time to investigate false alarm
    VALUE_PER_EARLY_DETECTION = 1400  # USD - value of early warning
    DOWNTIME_COST_PER_HOUR = 10000  # USD - cost of system downtime
    
    print(f"\n💰 Cost Assumptions:")
    print(f"   False Positive:       ${COST_PER_FP:,} (investigation time)")
    print(f"   Early Detection:      ${VALUE_PER_EARLY_DETECTION:,} (prevention value)")
    print(f"   Downtime per hour:    ${DOWNTIME_COST_PER_HOUR:,}")
    
    baseline_name = 'LAnoBERT (baseline)' if 'LAnoBERT (baseline)' in results else list(results.keys())[0]
    baseline = results[baseline_name]
    
    print(f"\n📊 Comparison vs {baseline_name}:\n")
    
    for model_name, metrics in results.items():
        if model_name == baseline_name:
            continue
        
        # Calculate savings
        baseline_fp = baseline.get('fp', 0)
        model_fp = metrics.get('fp', 0)
        fp_reduction = baseline_fp - model_fp
        fp_savings = fp_reduction * COST_PER_FP
        
        # Early detection value
        baseline_tp = baseline.get('tp', 0)
        model_tp = metrics.get('tp', 0)
        baseline_ewr = baseline.get('ewr', 0)
        model_ewr = metrics.get('ewr', 0)
        
        early_detections_increase = (model_tp * model_ewr) - (baseline_tp * baseline_ewr)
        early_detection_value = early_detections_increase * VALUE_PER_EARLY_DETECTION
        
        # Lead time improvement (downtime reduction)
        baseline_dlt = baseline.get('dlt_mean', 0)
        model_dlt = metrics.get('dlt_mean', 0)
        dlt_improvement = model_dlt - baseline_dlt  # seconds
        dlt_value = (dlt_improvement / 3600) * DOWNTIME_COST_PER_HOUR * model_tp
        
        total_value = fp_savings + early_detection_value + dlt_value
        
        print(f"{model_name}:")
        print("-" * 70)
        print(f"  FP Reduction:         {fp_reduction:>8,} alerts × ${COST_PER_FP} = ${fp_savings:>12,.2f}")
        print(f"  Early Detections:     {early_detections_increase:>8.1f} cases  × ${VALUE_PER_EARLY_DETECTION} = ${early_detection_value:>12,.2f}")
        print(f"  Lead Time Value:      {dlt_improvement:>8.1f}s improvement     = ${dlt_value:>12,.2f}")
        print(f"  {'─' * 68}")
        print(f"  Total Value:                                   = ${total_value:>12,.2f}")
        print()
    
    print("=" * 70)
    print("Note: These are estimates based on industry averages.")
    print("Actual ROI depends on specific operational context.")
    print("=" * 70)
else:
    print("⚠️  No results for ROI analysis")

## 7. Final Summary

In [ ]:
# Generate final summary
if results:
    print("=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    
    print(f"\n📊 Evaluated Models: {len(results)}")
    print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Find best model for each metric
    print("\n🏆 Best Performance:")
    
    metrics_to_rank = [
        ('F1-Score', 'f1', 'max'),
        ('AUROC', 'auroc', 'max'),
        ('False Positives', 'fp', 'min'),
        ('FPR', 'fpr', 'min'),
        ('DLT', 'dlt_mean', 'max'),
        ('EWR', 'ewr', 'max'),
    ]
    
    for metric_name, metric_key, order in metrics_to_rank:
        values = [(name, metrics.get(metric_key)) for name, metrics in results.items() 
                 if metrics.get(metric_key) is not None]
        
        if values:
            if order == 'max':
                best_model, best_val = max(values, key=lambda x: x[1])
            else:
                best_model, best_val = min(values, key=lambda x: x[1])
            
            if metric_key == 'fp':
                print(f"   {metric_name:<20} {best_model:<30} {int(best_val):>10,}")
            elif metric_key in ['fpr', 'ewr']:
                print(f"   {metric_name:<20} {best_model:<30} {best_val*100:>10.4f}%")
            elif metric_key == 'dlt_mean':
                print(f"   {metric_name:<20} {best_model:<30} {best_val:>10.1f}s")
            else:
                print(f"   {metric_name:<20} {best_model:<30} {best_val:>10.6f}")
    
    # Overall recommendation
    print("\n📋 Recommendation:")
    print("   Based on the comparison, the best model depends on priority:")
    print("\n   • Maximize F1/AUROC → Use the model with highest F1")
    print("   • Minimize False Positives → Use the model with lowest FP")
    print("   • Early Detection → Use the model with highest DLT + EWR")
    print("   • Balanced → Consider Phase 2 (Projection Head) if available")
    
    print("\n" + "=" * 70)
    print("✅ COMPARISON COMPLETE")
    print("=" * 70)
else:
    print("⚠️  No results to summarize")

## 8. Export Complete Report

In [ ]:
# Export comprehensive comparison report
if results:
    report = {
        'timestamp': datetime.now().isoformat(),
        'num_models': len(results),
        'models': list(results.keys()),
        'results': results,
        'comparison_matrix': df.to_dict() if 'df' in locals() else None,
    }
    
    # Save JSON
    os.makedirs('outputs/comparison', exist_ok=True)
    output_json = 'outputs/comparison/complete_comparison.json'
    with open(output_json, 'w') as f:
        json.dump(report, f, indent=2, default=str)
    
    print(f"✅ Report exported to: {output_json}")
    
    # Save CSV
    if 'df' in locals():
        output_csv = 'outputs/comparison/comparison_table.csv'
        df.to_csv(output_csv)
        print(f"✅ Table exported to: {output_csv}")
    
    print("\n📂 All outputs saved to: outputs/comparison/")
    print("   • complete_comparison.json (full data)")
    print("   • comparison_table.csv (metrics table)")
    print("   • performance_comparison.png (charts)")
    print("   • improvement_heatmap.png (heatmap)")
else:
    print("⚠️  No results to export")